# Aggregated signal analysis

This notebook performs a statistical analysis of the aggregated seismic acceleration signal, obtained by combining all 2,304,000 samples from the 48 truncated signals into a single time series. The analysis mirrors the structure of notebook 03 but treats the entire aggregated dataset as a single signal, and compares results across distance groups (Near, Mid, Far).

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import logging
from src import (
    set_plot_style,
    plot_empirical_pdfs,
    gaussian_fit_analysis,
    heavy_tail_assessment,
    heavy_tail_to_latex
)
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

## 2. Data loading

Preprocessed aggregated acceleration data are loaded from the parquet file 
produced in notebook 02. Only signals with at least 48000 samples are included, 
all truncated to exactly 48000 samples. Each signal has been baseline-corrected 
and normalized by its standard deviation, so that $\bar{a} = 0$ and $\sigma_a = 1$.

In [ ]:
# Load preprocessed data
df_acc_pooled= pd.read_parquet('../data/processed/02_signals/acc_preprocessed_pdf.parquet')
print("Shape:", df_acc_pooled.shape)
print(df_acc_pooled.head())
# → Dataframe completo, analizza con df_pooled['acceleration_normalized']

Shape: (2304000, 4)
                                       file  sample  acceleration  \
0  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       0 -6.666667e-10   
1  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       1 -6.666667e-10   
2  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       2 -6.666667e-10   
3  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       3 -6.666667e-10   
4  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       4 -6.666667e-10   

   acceleration_normalized  
0            -3.401661e-08  
1            -3.401661e-08  
2            -3.401661e-08  
3            -3.401661e-08  
4            -3.401661e-08  


In [8]:
# Figures output directory
FIGURES_DIR = Path('../figures/04_aggregated')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 4. PDF analysis

Empirical probability density functions and distribution fits are computed on the full aggregated dataset. 

### 4.1 Aggregated empirical PDF

Empirical PDF of the full aggregated dataset, combining all 48 signals.

Saved: 1/1 PDFs
All PDFs saved successfully!


### 4.2 Gaussian fit and normality assessment

A Gaussian distribution $\mathcal{N}(\mu, \sigma^2)$ is fitted to the 
full aggregated signal. The Anderson-Darling test is used to formally 
assess normality, and kurtosis and skewness are computed as quantitative 
measures of the shape of the distribution:

$$\kappa = \frac{\mu_4}{\sigma^4} - 3, \qquad \gamma = \frac{\mu_3}{\sigma^3}$$

A Q-Q plot against the Gaussian distribution is produced to visualize 
deviations from normality.

In [11]:
# Gaussian fit — global signal
df_gaussian_results_agg = gaussian_fit_analysis(df_global, bins=100, log_scale=True, output_dir=FIGURES_DIR / 'gaussian_fit')

Saved: 1/1 individual plots
All individual plots saved successfully!
Non-Gaussian signals (AD test, p<5%): 1/1


In [12]:
# Display Gaussian fit results
pd.set_option('display.max_rows', None)
display(df_gaussian_results_agg[['station', 'stream', 'kurtosis', 'skewness', 'ad_statistic', 'ad_critical_5pct', 'non_gaussian']])
pd.reset_option('display.max_rows')

,station,stream,kurtosis,skewness,ad_statistic,ad_critical_5pct,non_gaussian
0,ALL,HNA,57.5772,-0.0851,439425.8854,0.787,True


In [13]:
# Save Gaussian fit results
try:
    df_gaussian_results_agg.to_parquet('../data/processed/gaussian_fit_results_agg.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 4.3 Heavy-tail assessment

Gaussian, Laplace, Student-t and Lévy stable distributions are fitted 
to the full aggregated signal. The best fit is selected using the Akaike 
Information Criterion:

$$\text{AIC} = 2k - 2\ln\hat{L}$$

The power law exponent is estimated using the Hill estimator on the 
top 10\% of values:

$$\hat{\alpha} = \left( \frac{1}{k} \sum_{i=1}^{k} \ln \frac{x_{(i)}}{x_{(k+1)}} \right)^{-1}$$

In [ ]:
# Check partial results if available
try:
    df_partial = pd.read_parquet('../data/processed/heavy_tail_aggregated_partial.parquet')
    print(f"Partial results available (aggregated): {len(df_partial)} signals already processed")
except Exception:
    print("No partial results found (aggregated), starting from scratch")

In [ ]:
# Heavy-tail assessment — global signal
df_heavy_tail_agg = heavy_tail_assessment(df_global,
                                           output_dir=FIGURES_DIR / 'heavy_tail',
                                           resume=True)
try:
    df_heavy_tail_agg.to_parquet('../data/processed/heavy_tail_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

In [ ]:
print(df_heavy_tail_agg.columns.tolist())

### 5.4 Comparison across distance groups

Moment scaling analysis is performed separately for each distance group 
to investigate whether the scaling exponents $\zeta(q)$ and the linearity 
of the scaling change with epicentral distance.

In [ ]:
df_moments_by_group = {}
df_exponents_by_group = {}
df_linearity_by_group = {}
df_piecewise_by_group = {}

for group, df_g in [('Near', df_near), ('Mid', df_mid), ('Far', df_far)]:

    # Moments
    df_moments_by_group[group] = compute_moment_scaling(df_g, q_values, tau_values)

    # Scaling exponents
    df_exponents_by_group[group] = compute_scaling_exponents(
        df_moments_by_group[group],
        output_dir=FIGURES_DIR / f'scaling_{group.lower()}'
    )

    # Linearity check
    df_linearity_by_group[group] = test_scaling_linearity(
        df_exponents_by_group[group],
        output_dir=FIGURES_DIR / f'scaling_{group.lower()}'
    )

    # Piecewise fit
    df_piecewise_by_group[group] = fit_piecewise_scaling(
        df_exponents_by_group[group],
        output_dir=FIGURES_DIR / f'scaling_{group.lower()}'
    )

    try:
        df_exponents_by_group[group].to_parquet(
            f'../data/processed/scaling_exponents_{group.lower()}.parquet', index=False)
        df_piecewise_by_group[group].to_parquet(
            f'../data/processed/scaling_piecewise_{group.lower()}.parquet', index=False)
        print(f"Saved {group} results successfully!")
    except Exception as e:
        print(f"Error saving {group} results: {e}")

# Comparison plot
fig, ax = plt.subplots(figsize=(8, 6))
for i, group in enumerate(groups):
    df_mean = df_exponents_by_group[group].groupby('q')['zeta'].mean().reset_index()
    ax.plot(df_mean['q'], df_mean['zeta'], 'o-', color=group_colors[i],
            linewidth=1.5, markersize=5, label=group)
q_arr = np.array(q_values)
ax.plot(q_arr, 0.5 * q_arr, '--', color='gray', linewidth=1,
        label='Normal diffusion ($\\zeta(q) = q/2$)')
ax.set_xlabel('$q$')
ax.set_ylabel('$\\zeta(q)$')
ax.set_title('Mean scaling exponents by distance group')
ax.legend(title='Distance group')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'scaling_comparison_by_group.pdf', bbox_inches='tight')
plt.show()

### 6.3 Comparison across distance groups

Displacement autocorrelation functions and scaling exponents $\beta$ 
are computed separately for each distance group to investigate whether 
the temporal correlation structure of the signals changes with epicentral 
distance.

In [ ]:
df_autocorr_by_group = {}
C_matrices_by_group = {}
df_autocorr_scaling_by_group = {}

for group, df_g in [('Near', df_near), ('Mid', df_mid), ('Far', df_far)]:

    df_autocorr_by_group[group], C_matrices_by_group[group] = \
        compute_displacement_autocorrelation(
            df_g, output_dir=FIGURES_DIR / f'autocorrelation_{group.lower()}')

    df_autocorr_scaling_by_group[group] = analyze_autocorrelation_scaling(
        C_matrices_by_group[group],
        output_dir=FIGURES_DIR / f'autocorrelation_{group.lower()}')

    try:
        df_autocorr_scaling_by_group[group].to_parquet(
            f'../data/processed/autocorr_scaling_{group.lower()}.parquet', index=False)
        print(f"Saved {group} results successfully!")
    except Exception as e:
        print(f"Error saving {group} results: {e}")

# Comparison plot
fig, ax = plt.subplots(figsize=(8, 5))
for i, group in enumerate(groups):
    df_group_scaling = df_autocorr_scaling_by_group[group]
    ax.bar(
        [j + i * 0.25 for j in range(len(df_group_scaling))],
        df_group_scaling['beta'],
        width=0.25,
        color=group_colors[i],
        label=group
    )
ax.axhline(1, color='gray', linewidth=0.8, linestyle='--',
           label='$\\beta = 1$ (normal diffusion)')
ax.set_ylabel('$\\beta$')
ax.set_title('Autocorrelation scaling exponent by distance group')
ax.legend(title='Distance group')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'autocorr_scaling_by_group.pdf', bbox_inches='tight')
plt.show()